In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, lit, upper, month, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("Orders Analysis") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "4g") \
    .getOrCreate()

schema = StructType([
    StructField("Kraj", StringType(), True),
    StructField("Sprzedawca", StringType(), True),
    StructField("Data_zamowienia", StringType(), True),
    StructField("idZamowienia", IntegerType(), True),
    StructField("Utarg", StringType(), True)
])

df_orders = spark.read.csv(
    "zamowienia.txt",
    sep=";",
    header=True,
    schema=schema,
    encoding="utf-8"
)

df_orders = df_orders \
    .withColumn("Sprzedawca", regexp_replace("Sprzedawca", "[^\x00-\x7F]", "")) \
    .withColumn("Utarg", regexp_replace("Utarg", r"[^\d,]", "").cast("double") / 100) \
    .withColumn("Data_zamowienia", col("Data_zamowienia")) \
    .withColumn("idZamowienia", col("idZamowienia").cast("int"))

df_orders.write.bucketBy(8, "idZamowienia").sortBy("Utarg").mode("overwrite").saveAsTable("orders_bucketed")

agg_original = df_orders.groupBy("Kraj").agg({"Utarg": "sum"})
agg_original.show()

agg_bucketed = spark.sql("SELECT Kraj, SUM(Utarg) as Total_Utarg FROM orders_bucketed GROUP BY Kraj")
agg_bucketed.show()

df_orders.write.partitionBy("Kraj", "Sprzedawca").mode("overwrite").csv("partitioned_orders")

agg_partitioned = spark.read.csv("partitioned_orders", header=True, inferSchema=True)
agg_partitioned = agg_partitioned.groupBy("Kraj").agg({"Utarg": "sum"})
agg_partitioned.show()

subset1 = df_orders.withColumn("month", month(col("Data_zamowienia")))

subset2 = df_orders.withColumn("netto_utarg", col("Utarg") / 1.23)

subset3 = df_orders.withColumn("Sprzedawca", upper(col("Sprzedawca")))

subset4 = df_orders.withColumn("waluta", lit("PLN"))

subset1.createOrReplaceTempView("orders_subset1")
subset2.write.parquet("orders_subset2.parquet", mode="overwrite")
subset3.write.csv("orders_subset3.csv", mode="overwrite", header=True)
subset4.write.json("orders_subset4.json", mode="overwrite")

query = """
SELECT o.idZamowienia, o.Kraj, o.Sprzedawca, o.Data_zamowienia, o.Utarg,
       s1.month, s2.netto_utarg, s3.Sprzedawca as Sprzedawca_Upper, s4.waluta
FROM orders_subset1 o
JOIN parquet.`orders_subset2.parquet` s2 ON o.idZamowienia = s2.idZamowienia
JOIN csv.`orders_subset3.csv` s3 ON o.idZamowienia = s3.idZamowienia
JOIN json.`orders_subset4.json` s4 ON o.idZamowienia = s4.idZamowienia
"""
joined_data = spark.sql(query)
joined_data.show()